# Notebook 3 — Contextual Retrieval with Metadata and Parent-Child Chunks

This notebook adds a richer context layer on top of the retrieval pipeline from the previous notebooks.

The main ideas introduced here:

**Metadata schema** — every chunk now carries structured metadata (source type, parent ID, knowledge object type, file name, page number, language). This is defined using Pydantic so the data is always validated.

**Parent-child chunking** — small chunks are linked to the larger section they belong to. When a child chunk is retrieved, its parent context is also included. This gives the LLM more context without sending the whole document.

**Multi-source ingestion** — GitHub repositories, PDFs, and plain text notes are all supported with one consistent data model.

**Knowledge routing** — queries are routed to the most relevant source type before retrieval begins.

**How to run:** run all cells top to bottom. Add repo URLs, PDF paths, or text in the ingestion cell. Ask questions using the Gradio interface at the bottom.

**API key needed:** `GOOGLE_API_KEY` in Colab Secrets (for Gemini).


In [ ]:
!pip install -q --upgrade \
gitpython \
sentence-transformers \
qdrant-client \
rank-bm25 \
google-generativeai \
pydantic \
tiktoken \
pypdf \
gradio==5.35.0 \
tree-sitter==0.20.4 \
tree-sitter-languages==1.10.2

In [ ]:
import numpy as np

print("NumPy:", np.__version__)

from sentence_transformers import SentenceTransformer

print("Sentence Transformers OK")

from qdrant_client import QdrantClient

print("Qdrant OK")

import gradio

print("Gradio OK")

from tree_sitter_languages import get_parser

print("Tree-Sitter OK")

import google.generativeai as genai

print("Gemini SDK OK")

print("Environment Ready")

NumPy: 1.26.4
Sentence Transformers OK
Qdrant OK
Gradio OK
Tree-Sitter OK
Gemini SDK OK
Environment Ready


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Step 1 — Unified Metadata Schema with Pydantic

Every chunk in this notebook has a structured metadata object. This is built with Pydantic so if a required field is missing or the wrong type, you get a clear error right away.

What each field stores:

- `id` — unique ID for this chunk
- `parent_id` — ID of the parent chunk (empty if this is a top-level chunk)
- `source_type` — where this came from: `repo`, `pdf`, or `text`
- `knowledge_object` — what kind of thing this is: `file`, `class`, `function`, `page`, `note`
- `document_name` — which file or document this came from
- `page_number` — page number for PDF chunks
- `language` — programming language for code chunks

This metadata is stored alongside the embedding in Qdrant. It makes filtering, routing, and context expansion much easier.


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
import uuid


# Chunk Metadata
class ChunkMetadata(BaseModel):

    # Source Information

    source: str

    chunk_type: Optional[str] = None
    language: Optional[str] = None

    # Repository Metadata

    repository: Optional[str] = None
    file_path: Optional[str] = None

    class_name: Optional[str] = None
    function_name: Optional[str] = None

    line_start: Optional[int] = None
    line_end: Optional[int] = None

    imports: List[str] = Field(default_factory=list)

    # Parent / Child Retrieval

    parent_id: Optional[str] = None

    # PDF Metadata

    page_number: Optional[int] = None
    document_title: Optional[str] = None

    # Notes Metadata

    note_title: Optional[str] = None

    # Knowledge Routing

    knowledge_object: Optional[str] = None
    # repo
    # pdf
    # notes

    # Future GraphRAG

    related_chunks: List[str] = Field(default_factory=list)

    entities: List[str] = Field(default_factory=list)

    tags: List[str] = Field(default_factory=list)


# Chunk Object


class CodeChunk(BaseModel):

    id: str = Field(default_factory=lambda: str(uuid.uuid4()))

    content: str

    metadata: ChunkMetadata


print("Metadata models ready.")

Metadata models ready.


## Step 2 — Parent-Child Ingestion Pipeline

This is the key feature of this notebook.

**The problem with flat chunks:**

If a class has three methods and you only retrieve one method chunk, the answer is incomplete. The LLM does not know which class the method belongs to.

**The solution — parent-child linking:**

```
Parent:  UserService class  (stores a summary of what the class does)
  ├── Child: create_user() method
  ├── Child: get_user() method
  └── Child: delete_user() method
```

When `create_user()` is retrieved, the system also fetches the `UserService` parent context and includes both in the answer. The LLM now understands the local chunk AND the bigger picture it belongs to.

This works for all three source types:
- **Repos** — files are parents, classes/functions are children
- **PDFs** — the document is the parent, pages are children
- **Notes** — the note collection is the parent, individual notes are children


In [ ]:
from pathlib import Path
from typing import List
from pypdf import PdfReader
import uuid

# Repository Ingestion


def parse_code_relations(
    file_path: Path, repo_path: Path, repo_name: str
) -> List[CodeChunk]:

    chunks = []

    rel_path = str(file_path.relative_to(repo_path))

    code = file_path.read_text(encoding="utf-8", errors="ignore")

    imports = [
        line.strip()
        for line in code.splitlines()
        if (line.startswith("import ") or line.startswith("from "))
    ]

    parent_id = str(uuid.uuid4())

    # Parent Module Chunk

    parent_chunk = CodeChunk(
        id=parent_id,
        content=code[:1500],
        metadata=ChunkMetadata(
            source="repo",
            knowledge_object="repo",
            repository=repo_name,
            file_path=rel_path,
            imports=imports,
            chunk_type="module",
        ),
    )

    chunks.append(parent_chunk)

    # AST Child Chunks

    try:

        ast_chunks = ast_chunk_code(code, "python")

        for node in ast_chunks:

            chunks.append(
                CodeChunk(
                    content=node["content"],
                    metadata=ChunkMetadata(
                        source="repo",
                        knowledge_object="repo",
                        repository=repo_name,
                        file_path=rel_path,
                        class_name=(
                            node["name"] if "class" in node["node_type"] else None
                        ),
                        function_name=(
                            node["name"] if "function" in node["node_type"] else None
                        ),
                        line_start=node["start_line"],
                        line_end=node["end_line"],
                        parent_id=parent_id,
                        chunk_type=node["node_type"],
                    ),
                )
            )

    except Exception as e:

        print(f"AST parsing failed: {file_path}")

        print(e)

    return chunks


# PDF Ingestion


def parse_pdf_relations(pdf_path: Path) -> List[CodeChunk]:

    chunks = []

    reader = PdfReader(str(pdf_path))

    parent_id = str(uuid.uuid4())

    first_page = ""

    if len(reader.pages):

        first_page = reader.pages[0].extract_text() or ""

    parent_chunk = CodeChunk(
        id=parent_id,
        content=first_page[:1500],
        metadata=ChunkMetadata(
            source="pdf",
            knowledge_object="pdf",
            document_title=pdf_path.name,
            chunk_type="document",
        ),
    )

    chunks.append(parent_chunk)

    for page_num, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        if not text.strip():
            continue

        sections = chunk_text(text)

        for section in sections:

            chunks.append(
                CodeChunk(
                    content=section,
                    metadata=ChunkMetadata(
                        source="pdf",
                        knowledge_object="pdf",
                        document_title=pdf_path.name,
                        page_number=page_num,
                        parent_id=parent_id,
                        chunk_type="section",
                    ),
                )
            )

    return chunks


# Notes Ingestion


def parse_notes_relations(notes_text: str) -> List[CodeChunk]:

    chunks = []

    parent_id = str(uuid.uuid4())

    parent_chunk = CodeChunk(
        id=parent_id,
        content="User Notes",
        metadata=ChunkMetadata(
            source="notes",
            knowledge_object="notes",
            note_title="User Notes",
            chunk_type="document",
        ),
    )

    chunks.append(parent_chunk)

    paragraphs = [p.strip() for p in notes_text.split("\n\n") if p.strip()]

    for para in paragraphs:

        chunks.append(
            CodeChunk(
                content=para,
                metadata=ChunkMetadata(
                    source="notes",
                    knowledge_object="notes",
                    note_title="User Notes",
                    parent_id=parent_id,
                    chunk_type="note",
                ),
            )
        )

    return chunks


print("Ingestion functions ready.")

Ingestion functions ready.


In [ ]:
import tiktoken

# Tokenizer

enc = tiktoken.get_encoding("cl100k_base")


# Token Counter


def estimate_tokens(text):

    return len(enc.encode(text))


# Token-Based Chunking


def chunk_text(text, max_tokens=400, overlap=50):

    tokens = enc.encode(text)

    chunks = []

    start = 0

    while start < len(tokens):

        end = min(start + max_tokens, len(tokens))

        chunk_tokens = tokens[start:end]

        chunks.append(enc.decode(chunk_tokens))

        start += max_tokens - overlap

    return chunks


print("Text chunker ready.")

Text chunker ready.


In [ ]:
# Knowledge Base Builder


def build_knowledge_base(repo_urls=None, pdf_files=None, notes_text=""):

    repo_urls = repo_urls or []
    pdf_files = pdf_files or []

    chunks_db = []

    # Repositories

    for repo_url in repo_urls:

        try:

            repo_name = repo_url.rstrip("/").split("/")[-1]

            temp_dir = Path(tempfile.mkdtemp(prefix="traceiq_repo_"))

            print(f"Cloning {repo_name}")

            git.Repo.clone_from(repo_url, temp_dir, depth=1)

            py_files = list(temp_dir.rglob("*.py"))

            for file_path in py_files[:MAX_REPO_FILES]:

                if any(
                    part in {".git", ".venv", "venv", "__pycache__", "tests"}
                    for part in file_path.parts
                ):
                    continue

                try:

                    chunks_db.extend(
                        parse_code_relations(file_path, temp_dir, repo_name)
                    )

                except Exception as e:

                    print(f"Failed: {file_path}")

                    print(e)

            shutil.rmtree(temp_dir, ignore_errors=True)

        except Exception as e:

            print(f"Repository failed: {repo_url}")

            print(e)

    # PDFs

    for pdf_file in pdf_files:

        try:

            chunks_db.extend(parse_pdf_relations(Path(pdf_file)))

        except Exception as e:

            print(f"PDF failed: {pdf_file}")

            print(e)

    # Notes

    if notes_text.strip():

        chunks_db.extend(parse_notes_relations(notes_text))

    # Stats

    parents = sum(1 for c in chunks_db if c.metadata.parent_id is None)

    children = len(chunks_db) - parents

    print()

    print("Knowledge Base Ready")

    print(f"Total Chunks: {len(chunks_db)}")

    print(f"Parents: {parents}")

    print(f"Children: {children}")

    return chunks_db

In [ ]:
# test
from pathlib import Path

chunks_db = build_knowledge_base(
    repo_urls=["https://github.com/karpathy/minGPT"],
    pdf_files=["attention_is_all_you_need.pdf"],
    notes_text="""
    Transformer notes.
    Contextual retrieval notes.
    """,
)

Repository failed: https://github.com/karpathy/minGPT
name 'tempfile' is not defined
PDF failed: attention_is_all_you_need.pdf
[Errno 2] No such file or directory: 'attention_is_all_you_need.pdf'

Knowledge Base Ready
Total Chunks: 2
Parents: 1
Children: 1


In [ ]:
parents = sum(1 for c in chunks_db if c.metadata.parent_id is None)

children = len(chunks_db) - parents

print(parents)
print(children)

1
1


In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Embedding model ready.")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready.


In [ ]:
import numpy as np

if "chunks_db" not in globals():

    raise ValueError("Run build_knowledge_base() first.")

if "embed_model" not in globals():

    raise ValueError("Embedding model missing.")

print("Building embedding cache...")

EMBEDDINGS = embed_model.encode(
    [c.content for c in chunks_db], normalize_embeddings=True, show_progress_bar=True
)

print(f"Embedding cache ready: {len(EMBEDDINGS)} vectors")

Building embedding cache...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding cache ready: 2 vectors


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

if "chunks_db" not in globals():
    raise ValueError("chunks_db not found. Run build_knowledge_base() first.")

if "EMBEDDINGS" not in globals():
    raise ValueError("EMBEDDINGS not found. Run embedding cache cell first.")

print("Building Qdrant index...")

qc = QdrantClient(location=":memory:")

COLLECTION_NAME = "traceiq_context"

try:
    qc.delete_collection(COLLECTION_NAME)
except Exception:
    pass

qc.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDINGS.shape[1], distance=Distance.COSINE),
)

points = []

for idx, chunk in enumerate(chunks_db):

    points.append(
        PointStruct(
            id=idx,
            vector=EMBEDDINGS[idx].tolist(),
            payload={
                "chunk_id": chunk.id,
                "source": chunk.metadata.source,
                "knowledge_object": (chunk.metadata.knowledge_object),
                "chunk_type": (chunk.metadata.chunk_type),
            },
        )
    )

qc.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Qdrant indexed {len(points)} chunks")

Building Qdrant index...
Qdrant indexed 2 chunks


In [ ]:
# Knowledge Router


def route_query(query):

    q = query.lower()

    repo_terms = {
        "function",
        "class",
        "method",
        "implementation",
        "repository",
        "code",
        "api",
    }

    pdf_terms = {"paper", "research", "document", "pdf", "chapter", "author"}

    if any(term in q for term in repo_terms):
        return "repo"

    if any(term in q for term in pdf_terms):
        return "pdf"

    return "all"


print("Knowledge Router Ready")

Knowledge Router Ready


## Step 3 — Knowledge Routing

Before retrieval, the system looks at the query and decides which source type to search.

Each chunk has a `source_type` field (`repo`, `pdf`, or `text`). The router uses lightweight keyword matching to figure out the most relevant type:

- Code-related words → search repo chunks
- Document-related words → search PDF chunks
- Everything else → search all sources

This reduces noise. If you ask a code question, you do not waste time retrieving PDF chunks.


In [ ]:
CHUNK_LOOKUP = {chunk.id: chunk for chunk in chunks_db}
MAX_CONTEXT_TOKENS = 4000


def retrieve_context_with_budget(query, chunks, top_k=10):

    query_vec = embed_model.encode(query, normalize_embeddings=True)
    knowledge_filter = route_query(query)

    hits = qc.query_points(
        collection_name=COLLECTION_NAME, query=query_vec.tolist(), limit=top_k
    ).points

    chunk_lookup = {c.id: c for c in chunks}

    context_parts = []

    used_tokens = 0

    for hit in hits:

        child = chunks[hit.id]

        if (
            knowledge_filter != "all"
            and child.metadata.knowledge_object != knowledge_filter
        ):
            continue

        if child.metadata.parent_id is None:
            continue

        child_text = child.content

        child_tokens = estimate_tokens(child_text)

        if used_tokens + child_tokens > MAX_CONTEXT_TOKENS:
            break

        context_parts.append(f"""
=== CHILD ===

{child_text}
""")

        used_tokens += child_tokens

        parent = CHUNK_LOOKUP.get(child.metadata.parent_id)

        if parent:

            context_parts.append(f"""
=== PARENT ===

{parent.content}
""")

        siblings = [
            c
            for c in chunks
            if (c.metadata.parent_id == child.metadata.parent_id) and (c.id != child.id)
        ]

        if siblings:

            sibling_preview = "\n".join(s.content[:100] for s in siblings[:3])

            context_parts.append(f"""
=== SIBLINGS ===

{sibling_preview}
""")

    return "\n".join(context_parts)


print("Contextual Retrieval Ready")

Contextual Retrieval Ready


## Step 4 — Answer Generation with Gemini

After retrieval, the context package is sent to Gemini 2.5 Flash.

The context includes:
- The retrieved child chunks (most relevant pieces)
- Their parent contexts (the bigger sections they belong to)

The model is instructed to:
- Answer only from the provided context
- Not invent information
- Cite which file or page the answer came from

The token budget is enforced before this step — only chunks that fit within the limit are included.


In [ ]:
import os
import google.generativeai as genai

# Gemini Configuration

API_KEY = None

try:

    from google.colab import userdata

    API_KEY = userdata.get("GEMINI_API_KEY")

except Exception:

    API_KEY = os.getenv("GEMINI_API_KEY")


if not API_KEY:

    raise ValueError("GEMINI_API_KEY not found.")


genai.configure(api_key=str(API_KEY))


# Gemini Model


gemini_model = genai.GenerativeModel(model_name="gemini-2.5-flash")


# Answer Generation


def generate_budgeted_answer(query, context):

    prompt = f"""
You are TraceIQ.

Answer ONLY using the supplied context.

Rules:

1. Do not hallucinate.
2. Use parent-child relationships when helpful.
3. Use sibling context when relevant.
4. Cite repository files, documents, or notes when available.
5. If the answer is not present, respond exactly:

The retrieved context does not contain the answer.

Context:

{context}

Question:

{query}
"""

    try:

        response = gemini_model.generate_content(
            prompt, generation_config={"temperature": 0.1, "max_output_tokens": 1000}
        )

        return response.text

    except Exception as e:

        return f"Gemini Error: {e}"


print("Gemini Ready")

Gemini Ready


## TraceIQ Contextual RAG — Interface

Run this cell to launch the Gradio interface.

**What this system can do:**
- Multi-repository ingestion (add multiple GitHub repo URLs)
- Multi-PDF ingestion (add multiple PDF files)
- Notes knowledge base (paste any text)
- Parent-child retrieval (child chunk + parent context)
- Context expansion with siblings
- Dynamic context budgeting (respects token limits)
- Knowledge routing (sends queries to the right source)
- Gemini 2.5 Flash answer generation with source citations

Type a question and the system will search across all indexed sources and generate a grounded answer.


In [ ]:
import gradio as gr


def build_kb_ui(repo_urls, pdf_files, notes):

    global chunks_db

    repos = [r.strip() for r in repo_urls.split("\n") if r.strip()]

    pdf_paths = []

    if pdf_files:

        pdf_paths = [file.name for file in pdf_files]

    chunks_db = build_knowledge_base(
        repo_urls=repos, pdf_files=pdf_paths, notes_text=notes
    )
    if len(chunks_db) == 0:
        raise ValueError("No chunks were created.")

    return f"Knowledge Base Ready\n\nChunks: {len(chunks_db)}"


def ask_traceiq(question):

    if not question.strip():

        return ("Enter a question.", "")

    context = retrieve_context_with_budget(question, chunks_db)

    answer = generate_budgeted_answer(question, context)

    return (answer, context[:6000])


with gr.Blocks(title="TraceIQ Contextual RAG") as app:

    gr.Markdown("# TraceIQ Contextual RAG")

    with gr.Row():

        with gr.Column():

            repo_urls = gr.Textbox(
                label="Repository URLs", lines=5, placeholder="One GitHub repo per line"
            )

            pdf_files = gr.Files(label="Upload PDFs")

            notes = gr.Textbox(label="Notes", lines=6)

            build_btn = gr.Button("Build Knowledge Base")

            kb_status = gr.Textbox(label="Status")

        with gr.Column():

            question = gr.Textbox(label="Question", lines=4)

            ask_btn = gr.Button("Ask")

            answer = gr.Markdown()

    with gr.Accordion("Retrieved Context", open=False):

        context_view = gr.Textbox(lines=25)

    build_btn.click(
        fn=build_kb_ui, inputs=[repo_urls, pdf_files, notes], outputs=[kb_status]
    )

    ask_btn.click(fn=ask_traceiq, inputs=[question], outputs=[answer, context_view])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b233aec2d7907673db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
